# **DATA**

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplfinance as mpf

### Downloading data

In [ ]:
## Download raw data from yfinance
ticker = 'SPY'
print(f"Download {ticker} from yahoo finance.")
data = yf.download(ticker, start='2015-01-01', end='2025-10-01', progress=False, auto_adjust=False)
data.tail()
# plt.plot(data['Close'])

In [ ]:
## Handle MultiIndex (Flatten Columns)
if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
data.head()

In [ ]:
df = data.copy()

### Cleaning Data

In [ ]:
## Logically Inconsistent
# Rule A: Volume > 0
mask_vol = df['Volume'] >= 0

# Rule B: High >= Low
mask_hl = df['High'] >= df['Low']

# Rule C: Open and Close values must be bounded by High and Low, with a slight tolerance for rounding errors.
epsilon = 1e-4
mask_rng_close = (df['Close'] <= df['High'] + epsilon) & (df['Close'] >= df['Low'] - epsilon)
mask_rng_open  = (df['Open']  <= df['High'] + epsilon) & (df['Open']  >= df['Low'] - epsilon)

# Overall
valid_rows = mask_vol & mask_hl & mask_rng_close & mask_rng_open

invalid_count = len(df) - valid_rows.sum()
if invalid_count > 0:
    print(f"Detected {invalid_count} rows with logical errors (in raw data).")
    # Debug: Print a sample invalid row for verification
    # print("Sample invalid row:", df[~valid_rows].head(1))
    print('Illogical data removed.')
    df = df[valid_rows]
else:
    print('No illogical data found.')

In [ ]:
# Standardize column names
df['close'] = df['Adj Close'] # Use Adjusted Close to account for splits/dividends
df = df.rename(columns={
        'Open': 'open', 'High': 'high', 'Low': 'low', 'Volume': 'volume'
    })

# Filter/Keep only OHLCV columns
required_cols = ['open', 'high', 'low', 'close', 'volume']
df = df[required_cols]

df.tail()

In [ ]:
# Missing Values
## LOCF (Last Observation Carried Forward)
df = df.ffill()
## Delete NaN data
df = df.dropna()

In [ ]:
# Fat Tail Analysis: Check for extreme volatility
# Do not blindly remove outliers; flag them for investigation.
# Alert if daily fluctuation exceeds 20%.
# (Potential causes: M&A events, Earnings Surprises, or Flash Crashes)
daily_ret = df['close'].pct_change().abs()
extreme_moves = daily_ret[daily_ret > 0.20] # Daily returns > 20%

if not extreme_moves.empty:
    print(f"\n WARNING: Detected {len(extreme_moves)} days with extreme price volatility (> 20%).")
    print(extreme_moves.tail(3)) # Display the 3 most recent instances

In [ ]:
# Check for Zero Volume (Illiquidity)
# Low Liquidity Analysis
zero_vol_count = (df['volume'] == 0).sum()

if zero_vol_count > 0:
    print(f"Note: Detected {zero_vol_count} days with no trading activity (Volume=0).")
    # Crucial: Retain these rows to preserve time-series continuity for SMA calculations.
    # (Do not drop them, as gaps will disrupt the moving average window).

### Plotting

In [ ]:
# Configure Visualization Style
# 'yahoo': Replicates Yahoo Finance aesthetics (Green up / Red down candles)
my_style = mpf.make_mpf_style(base_mpf_style='yahoo', rc={'font.size': 10})

# Generate the Chart
# type='candle': Render as a Candlestick chart
# volume=True: Include a Volume panel at the bottom
df_short = df.loc['2025-06-01':'2025-10-01']

mpf.plot(df_short,
         type='candle',
         style=my_style,
         title=f'{ticker} OHLCV Chart (Adjusted)',
         ylabel='Price (USD)',
         ylabel_lower='Volume',
         volume=True,
         figsize=(14, 8),
         tight_layout=True)

In [ ]:
# Visualize Adjusted Close Price
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['close'], label=f'{ticker} Adj Close', color='blue', linewidth=1.5)

# Title: Clearly state the metric and the data status (Processed)
plt.title(f'{ticker} Adjusted Close Price History (Processed: 2015-2025)', fontsize=14)
plt.xlabel('Date', fontsize=12)      # Or 'Year'
plt.ylabel('Price (USD)', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, which='both', linestyle='--', linewidth=0.5)

plt.show()

# **TREND-FOLLOWING STRATEGY**
## SMA (Simple Moving Average)
- The N-day SMA is the average close price of the past N days.
- Formula: $SMA_{N} = \frac{1}{N} \sum_{i=0}^{N-1} C_{t-i}$
- **Drawback:** It treats all prices of the past N days equally, failing to give more weight to recent data.

In [ ]:
# Calculate Moving Averages (Short, Medium, and Long-term trends)
df['SMA_10'] = df['close'].rolling(window=10).mean()
df['SMA_50'] = df['close'].rolling(window=50).mean()
df['SMA_200'] = df['close'].rolling(window=200).mean()

# Subset data for plotting
df_plot = df.loc['2025-01-01':'2025-10-30'].copy()

plt.figure(figsize=(14, 7))

# Plot the underlying price (faded gray for better contrast with SMAs)
plt.plot(df_plot.index, df_plot['close'], label=f'{ticker} Adj Close', color='gray', linewidth=1.5, alpha=0.7)

# Plot the SMAs
plt.plot(df_plot.index, df_plot['SMA_10'], label='SMA 10 (Short-term)', color='blue', linewidth=2)
plt.plot(df_plot.index, df_plot['SMA_50'], label='SMA 50 (Mid-term)', color='red', linewidth=2)
plt.plot(df_plot.index, df_plot['SMA_200'], label='SMA 200 (Long-term)', color='orange', linewidth=2)

# Chart Styling
plt.title(f'{ticker} Trend Analysis: Moving Averages (SMA 10/50/200)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, which='both', linestyle='--', linewidth=0.5)

plt.show()

- If the market closes above its 200-day SMA line, the market is in uptrend; if the market closes below its 200-day SMA line, the market is in downtrend

### Crossover Strategy
The strategy utilizes the interaction between the short-term trend (MA 50) and the long-term trend (MA 200) to generate signals:

* **📈 Long Entry (Buy Signal):** Triggered when the **MA 50 crosses above the MA 200**. This pattern is technically known as the **"Golden Cross"**, signaling the inception of a long-term **bullish** trend.
* **📉 Short Entry / Exit (Sell Signal):** Triggered when the **MA 50 crosses below the MA 200**. This pattern is referred to as the **"Death Cross"**, indicating a shift toward a **bearish** market structure.

In [ ]:
def strategy_sma_crossover(data_input):
    """
    Implements a classic SMA Crossover Strategy (Golden Cross / Death Cross).
    Returns a DataFrame with generated signals and backtesting metrics.
    """
    df_sma = data_input.copy()

    # 1. Calculate Moving Averages (Indicators)
    # SMA 50 (Short-term) and SMA 200 (Long-term)
    df_sma['SMA_50'] = df_sma['close'].rolling(window=50).mean()
    df_sma['SMA_200'] = df_sma['close'].rolling(window=200).mean()

    # 2. Generate Trading Signals
    # Signal = 1: Long Position (Buy/Hold) when SMA 50 > SMA 200 (Golden Cross)
    # Signal = 0: Cash Position (Sell/Exit) when SMA 50 <= SMA 200
    df_sma['Signal'] = 0
    df_sma.loc[df_sma['SMA_50'] > df_sma['SMA_200'], 'Signal'] = 1

    # 3. Calculate Strategy Performance
    # Market Return: Daily percentage change of the asset
    df_sma['Market_Return'] = df_sma['close'].pct_change()

    # Strategy Return: Return based on the signal
    # CRITICAL: .shift(1) is used to simulate executing the trade on the NEXT day
    # based on TODAY's signal. This prevents 'Look-Ahead Bias'.
    df_sma['Strategy_Return'] = df_sma['Signal'].shift(1) * df_sma['Market_Return']

    # 4. Calculate Cumulative Returns (Equity Curve)
    # Uses cumulative product to simulate compound growth
    df_sma['Cumulative_Market'] = (1 + df_sma['Market_Return']).cumprod()
    df_sma['Cumulative_Strategy'] = (1 + df_sma['Strategy_Return']).cumprod()

    return df_sma

In [ ]:
# --- VISUALIZE TRADING SIGNALS ---

results = strategy_sma_crossover(df)

# Identify Entry and Exit points
# Use .diff() to detect signal transitions
# diff =  1: Signal shifts from 0 to 1 (LONG Entry / Golden Cross)
# diff = -1: Signal shifts from 1 to 0 (EXIT/SHORT / Death Cross)
results['Position_Change'] = results['Signal'].diff()

# Filter specific timestamps where trades are executed
buy_signals = results[results['Position_Change'] == 1]
sell_signals = results[results['Position_Change'] == -1]

# 3. Plotting the Chart
plt.figure(figsize=(14, 8))

# Plot underlying price (faded for clarity)
plt.plot(results.index, results['close'], label='Close Price', color='black', alpha=0.3)

# Plot SMA 50 (Short-term Trend)
plt.plot(results.index, results['SMA_50'], label='SMA 50', color='orange', linewidth=1.5)

# Plot SMA 200 (Long-term Trend)
plt.plot(results.index, results['SMA_200'], label='SMA 200', color='blue', linewidth=2)

# Marker: BUY Signals (Up Triangle)
plt.scatter(buy_signals.index, buy_signals['close'],
            marker='^', color='green', s=150, zorder=5, label='BUY (Golden Cross)')

# Marker: SELL Signals (Down Triangle)
plt.scatter(sell_signals.index, sell_signals['close'],
            marker='v', color='red', s=150, zorder=5, label='SELL (Death Cross)')

plt.title('SMA Crossover Strategy: Buy & Sell Signal Execution', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)

plt.show()

### Backtest

In [ ]:
def calculate_advanced_metrics(results, risk_free_rate=0.0):
    # ==========================================
    # PART 1: TIME-SERIES METRICS (Based on daily equity curve)
    # ==========================================

    # 1. CAGR (Compound Annual Growth Rate)
    # Calculates the smoothed annualized growth rate
    days = (results.index[-1] - results.index[0]).days
    total_return = results['Cumulative_Strategy'].iloc[-1]
    cagr = (total_return) ** (365.0 / days) - 1

    # 2. Max Drawdown (MDD)
    running_max = results['Cumulative_Strategy'].cummax()
    drawdown = (results['Cumulative_Strategy'] - running_max) / running_max
    max_drawdown = drawdown.min()

    # 3. Sharpe Ratio & Sortino Ratio
    daily_returns = results['Strategy_Return']
    excess_returns = daily_returns - (risk_free_rate / 252)

    # Sharpe (Total Volatility)
    sharpe_ratio = (excess_returns.mean() / excess_returns.std()) * (252 ** 0.5)

    # Sortino (Downside Volatility only)
    # Only penalizes returns that fall below the target (0)
    negative_returns = excess_returns[excess_returns < 0]
    downside_std = negative_returns.std()
    sortino_ratio = (excess_returns.mean() / downside_std) * (252 ** 0.5) if downside_std != 0 else 0

    # ==========================================
    # PART 2: TRADE-LEVEL METRICS (Based on individual trades)
    # ==========================================

    # Logic: Signal = 1 (Long/Hold), Signal = 0 (Cash/Exit)
    # We loop through the data to identify Entry (0->1) and Exit (1->0) points.

    trades = []
    in_position = False
    entry_price = 0
    entry_date = None

    for index, row in results.iterrows():
        # ENTRY Signal
        if row['Signal'] == 1 and not in_position:
            entry_price = row['close']
            entry_date = index
            in_position = True

        # EXIT Signal (or End of Data)
        elif (row['Signal'] == 0 and in_position) or (index == results.index[-1] and in_position):
            exit_price = row['close']
            pnl_pct = (exit_price - entry_price) / entry_price
            duration = (index - entry_date).days

            trades.append({
                'Entry_Date': entry_date,
                'Exit_Date': index,
                'PnL': pnl_pct,
                'Duration': duration
            })
            in_position = False

    # Convert list of trades to DataFrame for analysis
    trade_df = pd.DataFrame(trades)

    if not trade_df.empty:
        total_trades = len(trade_df)
        win_trades = trade_df[trade_df['PnL'] > 0]
        loss_trades = trade_df[trade_df['PnL'] <= 0]

        # Win Rate (Per Trade)
        win_rate = len(win_trades) / total_trades

        # Profit Factor (Gross Profit / Absolute Gross Loss)
        gross_profit = win_trades['PnL'].sum()
        gross_loss = abs(loss_trades['PnL'].sum())
        profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.inf

        # Average Return Per Trade
        avg_trade = trade_df['PnL'].mean()

        # Risk Reward Ratio (Avg Win / Avg Loss)
        avg_win = win_trades['PnL'].mean() if not win_trades.empty else 0
        avg_loss = abs(loss_trades['PnL'].mean()) if not loss_trades.empty else 0
        risk_reward = avg_win / avg_loss if avg_loss != 0 else 0

    else:
        # Fallback if no trades occurred
        total_trades = 0
        win_rate = 0
        profit_factor = 0
        avg_trade = 0
        risk_reward = 0

    # ==========================================
    # PART 3: PRINT REPORT
    # ==========================================
    print("=" * 40)
    print("     ADVANCED STRATEGY PERFORMANCE")
    print("=" * 40)
    print(f"1. RETURN METRICS")
    print(f"   - Total Return (Net): {total_return - 1:.2%}")
    print(f"   - CAGR (Annualized) : {cagr:.2%}")
    print("-" * 40)
    print(f"2. RISK METRICS")
    print(f"   - Max Drawdown      : {max_drawdown:.2%}")
    print(f"   - Sharpe Ratio      : {sharpe_ratio:.2f}")
    print(f"   - Sortino Ratio     : {sortino_ratio:.2f}")
    print("-" * 40)
    print(f"3. TRADE METRICS (Vital Stats)")
    print(f"   - Total Trades      : {total_trades}")
    print(f"   - Win Rate          : {win_rate:.2%} (Trades Won / Total)")
    print(f"   - Profit Factor     : {profit_factor:.2f} (Target > 1.5)")
    print(f"   - Risk/Reward Ratio : 1:{risk_reward:.2f}")
    print(f"   - Avg Return/Trade  : {avg_trade:.2%}")
    print("=" * 40)

    return trade_df

# --- USAGE EXAMPLE ---
results = strategy_sma_crossover(df)
trade_log = calculate_advanced_metrics(results)

In [ ]:
# Comparison
def compare_strategy_vs_buy_hold(results, risk_free_rate=0.0):
    results['Market_Return'] = results['close'].pct_change().fillna(0)
    results['Cumulative_Market'] = (1 + results['Market_Return']).cumprod()

    def get_metrics(returns_series, cumulative_series):
        # CAGR
        days = (results.index[-1] - results.index[0]).days
        total_ret = cumulative_series.iloc[-1]
        cagr = (total_ret) ** (365.0 / days) - 1

        # Max Drawdown
        running_max = cumulative_series.cummax()
        drawdown = (cumulative_series - running_max) / running_max
        mdd = drawdown.min()

        # Sharpe
        excess_ret = returns_series - (risk_free_rate / 252)
        sharpe = (excess_ret.mean() / excess_ret.std()) * (252 ** 0.5) if excess_ret.std() != 0 else 0

        return total_ret - 1, cagr, mdd, sharpe

    strat_total, strat_cagr, strat_mdd, strat_sharpe = get_metrics(results['Strategy_Return'], results['Cumulative_Strategy'])
    mkt_total, mkt_cagr, mkt_mdd, mkt_sharpe = get_metrics(results['Market_Return'], results['Cumulative_Market'])

    print("=" * 60)
    print(f"{'METRIC':<20} | {'STRATEGY':<15} | {'BUY & HOLD':<15}")
    print("-" * 60)
    print(f"{'Total Return':<20} | {strat_total:>14.2%} | {mkt_total:>14.2%}")
    print(f"{'CAGR (Yearly)':<20} | {strat_cagr:>14.2%} | {mkt_cagr:>14.2%}")
    print(f"{'Max Drawdown':<20} | {strat_mdd:>14.2%} | {mkt_mdd:>14.2%}")
    print(f"{'Sharpe Ratio':<20} | {strat_sharpe:>14.2f} | {mkt_sharpe:>14.2f}")
    print("=" * 60)

    running_max_strat = results['Cumulative_Strategy'].cummax()
    drawdown_strat = (results['Cumulative_Strategy'] - running_max_strat) / running_max_strat

    running_max_mkt = results['Cumulative_Market'].cummax()
    drawdown_mkt = (results['Cumulative_Market'] - running_max_mkt) / running_max_mkt
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
    ax1.plot(results.index, results['Cumulative_Strategy'], label='Strategy Equity', color='blue', linewidth=1.5)
    ax1.plot(results.index, results['Cumulative_Market'], label='Buy & Hold (Market)', color='gray', linestyle='--', alpha=0.7)
    
    ax1.set_title('Performance Analysis: Strategy vs. Buy & Hold', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Equity Growth (Start = 1.0)')
    ax1.legend(loc='upper left')
    ax1.grid(True, linestyle=':', alpha=0.6)

    ax2.plot(results.index, drawdown_mkt, label='Market Drawdown', color='gray', alpha=0.3, linestyle='--')
    ax2.fill_between(results.index, drawdown_mkt, 0, color='gray', alpha=0.1)
    ax2.plot(results.index, drawdown_strat, label='Strategy Drawdown', color='red', linewidth=1.2)
    ax2.fill_between(results.index, drawdown_strat, 0, color='red', alpha=0.3)
    
    ax2.set_ylabel('Drawdown (%)')
    ax2.set_xlabel('Date')
    ax2.legend(loc='lower right')
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.axhline(0, color='black', linewidth=0.8)

    plt.tight_layout()
    plt.show()

compare_strategy_vs_buy_hold(results)


📉 Why Pure SMA Strategies Often Fail

**1. The Cost of Lag (Latency Issue)**

* **Problem:** The SMA treats all data points equally, causing it to react sluggishly to sharp price reversals.
* **Consequence:** By the time the SMA confirms a trend change, the price has already moved significantly from the bottom. This results in **late entries** and missed profit potential during the initial phase of the trend.

**2. Whipsaws in Sideways Markets (False Signals)**

* **Problem:** In range-bound (sideways) markets where there is no clear trend, price action fluctuates around the average, triggering frequent crossover signals.
* **Consequence:** The strategy suffers from **"Buying High, Selling Low"** repeatedly. The capital is slowly eroded by a series of consecutive small losses (often referred to as "Death by a Thousand Cuts").

---

🚀 Optimization: Mitigating Lag

**Solution: Transitioning to EMA**
Switching to the **Exponential Moving Average (EMA)** is the first step in upgrading the model to address the "equal weighting" flaw of the standard SMA.

* **Theory:** Unlike the SMA, the EMA assigns **greater weight to the most recent price data**. This weighting scheme allows the indicator to react more rapidly to recent market changes, significantly reducing the inherent lag.
* **Formula:**
$$EMA_t = (P_t \times k) + (EMA_{t-1} \times (1 - k))$$
* $EMA_t$: The current EMA value.
* $P_t$: The current closing price.
* $EMA_{t-1}$: The previous EMA value.


*Where the smoothing factor is:* $$k = \frac{2}{N + 1}$$


## LAGS
### EMA (Exponential Moving Average)

In [ ]:
def strategy_ema(df):

    df_ema = df.copy()

    # --- 1. Calculate EMA Indicators ---
    # We use .ewm (Exponential Weighted Functions) to calculate the EMA.
    # adjust=False ensures recursive calculation: EMA_t = k * Price + (1-k) * EMA_t-1
    df_ema['EMA_50'] = df_ema['close'].ewm(span=50, adjust=False).mean()
    df_ema['EMA_200'] = df_ema['close'].ewm(span=200, adjust=False).mean()

    # --- 2. Generate Signals ---
    # Default Signal = 0 (Cash/Neutral)
    df_ema['Signal'] = 0

    # BUY Condition: Short-term EMA > Long-term EMA (Golden Cross)
    df_ema.loc[df_ema['EMA_50'] > df_ema['EMA_200'], 'Signal'] = 1

    # Identify Entry/Exit points for plotting purposes (diff: 1 = Buy, -1 = Sell)
    df_ema['Order'] = df_ema['Signal'].diff()

    # --- 3. Calculate Returns ---
    # Market Return (Benchmark / Buy & Hold)
    df_ema['Market_Return'] = df_ema['close'].pct_change()

    # Strategy Return
    # CRITICAL: We use .shift(1) to apply yesterday's signal to today's market return.
    # This prevents "Look-Ahead Bias" (you cannot trade on today's close price with today's signal).
    df_ema['Strategy_Return'] = df_ema['Signal'].shift(1) * df_ema['Market_Return']

    # --- 4. Calculate Cumulative Performance ---
    # Calculates the equity curve starting from a base of 1.0 (compounded)
    df_ema['Cumulative_Market'] = (1 + df_ema['Market_Return'].fillna(0)).cumprod()
    df_ema['Cumulative_Strategy'] = (1 + df_ema['Strategy_Return'].fillna(0)).cumprod()

    return df_ema

In [ ]:
# --- PLOT TRADING SIGNALS ---

# 1. Run the strategy
results2 = strategy_ema(df)

# 2. Identify Buy and Sell Entry Points
# Use .diff() to find where the Signal value changes
# diff =  1: Signal went from 0 to 1 (BUY - Golden Cross)
# diff = -1: Signal went from 1 to 0 (SELL - Death Cross/Exit)
results2['Position_Change'] = results2['Signal'].diff()

# Filter the dataframe to get only the trade execution days
buy_signals2 = results2[results2['Position_Change'] == 1]
sell_signals2 = results2[results2['Position_Change'] == -1]

# 3. Plot the Chart
plt.figure(figsize=(14, 8))

# Plot the underlying Asset Price (Close) - dimmed with alpha for clarity
plt.plot(results2.index, results2['close'], label='Price (Close)', color='black', alpha=0.3)

# Plot EMA 50 (Short-term / Fast)
plt.plot(results2.index, results2['EMA_50'], label='EMA 50', color='orange', linewidth=1.5)

# Plot EMA 200 (Long-term / Slow)
plt.plot(results2.index, results2['EMA_200'], label='EMA 200', color='blue', linewidth=2)

# Plot Buy Markers (Green Up-Triangle)
plt.scatter(buy_signals2.index, buy_signals2['close'],
            marker='^', color='green', s=150, zorder=5, label='BUY (Golden Cross)')

# Plot Sell Markers (Red Down-Triangle)
plt.scatter(sell_signals2.index, sell_signals2['close'],
            marker='v', color='red', s=150, zorder=5, label='SELL (Death Cross)')

# Chart Formatting
plt.title('EMA Strategy: Buy & Sell Signals', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend(loc='best') # Automatically finds the best spot for the legend
plt.grid(True, alpha=0.3) # Faint grid lines

plt.show()

### Backtest

In [ ]:
results_ema = strategy_ema(df)
metrics_ema = calculate_advanced_metrics(results_ema, risk_free_rate=0.0)

In [ ]:

# --- STEP 2: EVALUATION FUNCTION (Reusable) ---
def evaluate_strategy_vs_market(results, risk_free_rate=0.0):
    """
    Calculates financial metrics and plots Strategy vs. Market performance.
    """

    # 1. Define Metric Calculation Helper
    def get_metrics(returns, cumulative):
        days = (results.index[-1] - results.index[0]).days
        total_ret = cumulative.iloc[-1] - 1

        # CAGR (Compound Annual Growth Rate)
        if cumulative.iloc[-1] > 0:
            cagr = (cumulative.iloc[-1]) ** (365.0 / days) - 1
        else:
            cagr = 0

        # Max Drawdown
        running_max = cumulative.cummax()
        dd = (cumulative - running_max) / running_max
        mdd = dd.min()

        # Sharpe Ratio
        excess = returns - (risk_free_rate / 252)
        std_dev = excess.std()
        sharpe = (excess.mean() / std_dev) * (252 ** 0.5) if std_dev != 0 else 0

        return total_ret, cagr, mdd, sharpe

    # Calculate metrics for both Strategy and Market
    st_tot, st_cagr, st_mdd, st_sharpe = get_metrics(results['Strategy_Return'], results['Cumulative_Strategy'])
    mk_tot, mk_cagr, mk_mdd, mk_sharpe = get_metrics(results['Market_Return'], results['Cumulative_Market'])

    # 2. Calculate Trade-Level Statistics (Win Rate, Profit Factor)
    trades = []
    in_pos = False
    entry_p = 0

    # Loop to simulate trades based on signals
    for idx, row in results.iterrows():
        # ENTRY Logic (Signal 1)
        if row['Signal'] == 1 and not in_pos:
            entry_p = row['close']
            in_pos = True
        # EXIT Logic (Signal 0 or End of Data)
        elif (row['Signal'] == 0 and in_pos) or (idx == results.index[-1] and in_pos):
            exit_p = row['close']
            pnl = (exit_p - entry_p) / entry_p
            trades.append(pnl)
            in_pos = False

    trades_arr = np.array(trades)
    num_trades = len(trades_arr)

    # Win Rate calculation
    win_rate = np.sum(trades_arr > 0) / num_trades if num_trades > 0 else 0

    # Profit Factor calculation
    gross_profit = np.sum(trades_arr[trades_arr > 0])
    gross_loss = abs(np.sum(trades_arr[trades_arr <= 0]))
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else 999 # 999 indicates infinite profit factor (no loss)

    # 3. PRINT PERFORMANCE REPORT
    print("\n" + "="*60)
    print(f"{'EMA STRATEGY PERFORMANCE REPORT':^60}")
    print("="*60)
    print(f"{'METRIC':<20} | {'EMA STRATEGY':<15} | {'BUY & HOLD':<15}")
    print("-" * 60)
    print(f"{'Total Return':<20} | {st_tot:>14.2%} | {mk_tot:>14.2%}")
    print(f"{'CAGR':<20} | {st_cagr:>14.2%} | {mk_cagr:>14.2%}")
    print(f"{'Max Drawdown':<20} | {st_mdd:>14.2%} | {mk_mdd:>14.2%}")
    print(f"{'Sharpe Ratio':<20} | {st_sharpe:>14.2f} | {mk_sharpe:>14.2f}")
    print("-" * 60)
    print(f"{'TRADE STATISTICS':^60}")
    print("-" * 60)
    print(f"Total Trades      : {num_trades}")
    print(f"Win Rate          : {win_rate:.2%}")
    print(f"Profit Factor     : {profit_factor:.2f}")
    print("="*60 + "\n")

    # 4. PLOT DUAL CHARTS (Equity + Drawdown)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

    # --- Top Chart: Equity Curve ---
    ax1.plot(results.index, results['Cumulative_Strategy'], label='EMA Strategy', color='#1f77b4', linewidth=2)
    ax1.plot(results.index, results['Cumulative_Market'], label='Buy & Hold', color='gray', linestyle='--', alpha=0.6)
    ax1.set_title('Equity Curve: EMA Strategy vs Buy & Hold', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Growth (Start=1.0)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # --- Bottom Chart: Underwater Plot ---
    # Calculate drawdown series
    dd_strat = (results['Cumulative_Strategy'] - results['Cumulative_Strategy'].cummax()) / results['Cumulative_Strategy'].cummax()
    dd_mkt = (results['Cumulative_Market'] - results['Cumulative_Market'].cummax()) / results['Cumulative_Market'].cummax()

    # Plot Strategy Drawdown (Red area)
    ax2.plot(results.index, dd_strat, label='EMA Drawdown', color='red', linewidth=1)
    ax2.fill_between(results.index, dd_strat, 0, color='red', alpha=0.2)

    # Plot Market Drawdown (Gray line)
    ax2.plot(results.index, dd_mkt, label='Market Drawdown', color='gray', linestyle='--', alpha=0.3)

    ax2.set_title('Underwater Plot (Drawdown)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Drawdown %')

    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# --- STEP 3: EXECUTE ---
evaluate_strategy_vs_market(results_ema)

### DEMA (Double exponential moving average)
DEMA is a sophisticated mathematical formula designed to add "velocity" to the moving average, allowing it to catch up to the current price effectively (achieving a "zero lag" effect).Formula:

$$DEMA = 2 \times EMA_N - EMA(EMA_N)$$

Where:
* $EMA_N$: The First-order EMA of the price (the standard EMA).
* $EMA(EMA_N)$: The Second-order EMA (calculated by taking the EMA of the first $EMA_N$).
* The Logic Behind "Zero Lag":
   * In the equation $2 \times EMA_1 - EMA_2$: The $EMA_2$ is naturally slower and lags more than $EMA_1$.
* The difference $(EMA_1 - EMA_2)$ represents the "Lag Error" (the distance by which the smoothing calculation trails the price).
* To correct this, the formula adds this error term back to the original $EMA_1$ to "push" the indicator forward in time:

$$EMA_1 + (\text{Lag Error}) \rightarrow EMA_1 + (EMA_1 - EMA_2) = 2EMA_1 - EMA_2$$

In [ ]:
def calculate_dema(series, span):
    # Calculate EMA1 (First Smoothing)
    ema1 = series.ewm(span=span, adjust=False).mean()
    # Calculate EMA2 (Second Smoothing of EMA1)
    ema2 = ema1.ewm(span=span, adjust=False).mean()

    # Calculate Final DEMA
    dema = 2 * ema1 - ema2
    return dema


def strategy_dema(df):
    data = df.copy()

    # --- A. INDICATOR CALCULATION ---
    # 1. Calculate DEMA Lines
    data['DEMA_50'] = calculate_dema(data['close'], span=50)   # Fast Line
    data['DEMA_200'] = calculate_dema(data['close'], span=200) # Slow Line

    # --- B. SIGNAL GENERATION ---
    data['Signal'] = 0

    # Buy Condition: Short-term DEMA > Long-term DEMA (Golden Cross)
    # Since DEMA reacts faster, this signal should trigger earlier than standard EMA.
    data.loc[data['DEMA_50'] > data['DEMA_200'], 'Signal'] = 1

    # --- C. PERFORMANCE CALCULATION ---
    data['Market_Return'] = data['close'].pct_change()

    # Strategy Return: Apply yesterday's signal to today's market return (shift 1)
    data['Strategy_Return'] = data['Signal'].shift(1) * data['Market_Return']

    # Calculate Cumulative Returns (Equity Curve)
    data['Cumulative_Market'] = (1 + data['Market_Return'].fillna(0)).cumprod()
    data['Cumulative_Strategy'] = (1 + data['Strategy_Return'].fillna(0)).cumprod()

    # Calculate Drawdown for the Strategy
    running_peak = data['Cumulative_Strategy'].cummax()
    data['Drawdown'] = (data['Cumulative_Strategy'] - running_peak) / running_peak

    return data

In [ ]:
# --- Comparing lags in each MA ---

results3 = strategy_sma_crossover(df)
results2 = strategy_ema(df)
results = strategy_dema(df)

subset = results.tail(200)
subset2 = results2.tail(200)
subset3 = results3.tail(200)
plt.figure(figsize=(14, 8))

# 1. Giá
plt.plot(subset.index, subset['close'], color='black', alpha=0.3, label='Giá', linewidth=3)

# 2. So sánh Short MA (50 ngày)
plt.plot(subset3.index, subset3['SMA_50'], color='blue', linewidth=2, label='SMA 50 ')
plt.plot(subset2.index, subset2['EMA_50'], color='orange', linestyle='--', label='EMA 50 ')
plt.plot(subset.index, subset['DEMA_50'], color='red', linewidth=2, label='DEMA 50 ')

plt.title(f'Comparison lags in each MA{ticker}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# --- PLOT TRADING SIGNALS ---

# 1. Run the DEMA strategy
results_dema = strategy_dema(df)

# 2. Identify Buy and Sell Signals
# Use .diff() to detect changes in the 'Signal' column
# diff =  1: Signal changes from 0 to 1 (BUY - Golden Cross)
# diff = -1: Signal changes from 1 to 0 (SELL - Death Cross/Exit)
results_dema['Position_Change'] = results_dema['Signal'].diff()

# Filter the DataFrame to get only the specific entry/exit rows
buy_signals = results_dema[results_dema['Position_Change'] == 1]
sell_signals = results_dema[results_dema['Position_Change'] == -1]

# 3. Initialize the Plot
plt.figure(figsize=(14, 8))

# Plot the underlying Asset Price (Close) - set alpha to 0.3 to make indicators pop
plt.plot(results.index, results['close'], label='Price (Close)', color='black', alpha=0.3)

# Plot DEMA 50 (Short-term / Fast Line - Orange)
plt.plot(results.index, results['DEMA_50'], label='DEMA 50', color='orange', linewidth=1.5)

# Plot DEMA 200 (Long-term / Slow Line - Blue)
plt.plot(results.index, results['DEMA_200'], label='DEMA 200', color='blue', linewidth=2)

# Plot Buy Markers (Green Up-Triangle)
plt.scatter(buy_signals.index, buy_signals['close'],
            marker='^', color='green', s=150, zorder=5, label='BUY (Golden Cross)')

# Plot Sell Markers (Red Down-Triangle)
plt.scatter(sell_signals.index, sell_signals['close'],
            marker='v', color='red', s=150, zorder=5, label='SELL (Death Cross)')

# Chart Formatting
plt.title('DEMA Strategy: Buy & Sell Signals', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)

# Display the chart
plt.show()

### Backtest

In [ ]:
results_dema = strategy_dema(df)
metrics_dema = calculate_advanced_metrics(results_dema, risk_free_rate=0.0)

In [ ]:
def evaluate_dema_vs_market(results, risk_free_rate=0.0):
    """
    Calculates financial metrics and plots DEMA Strategy vs. Buy & Hold.
    """
    # 1. Prepare Data (Standardize column names)
    # Check if columns exist, handle naming variations
    col_strat_ret = 'Strategy_Return' if 'Strategy_Return' in results.columns else 'Strategy_Ret'
    col_mkt_ret = 'Market_Return' if 'Market_Return' in results.columns else 'Market_Ret'

    # Ensure Cumulative Market exists
    if 'Cumulative_Market' not in results.columns:
        results['Cumulative_Market'] = (1 + results[col_mkt_ret].fillna(0)).cumprod()

    # 2. Metric Calculation Helper
    def get_metrics(returns, cumulative):
        days = (results.index[-1] - results.index[0]).days
        total_ret = cumulative.iloc[-1] - 1

        # CAGR (Compound Annual Growth Rate)
        cagr = (cumulative.iloc[-1]) ** (365.0 / days) - 1 if cumulative.iloc[-1] > 0 else 0

        # Max Drawdown
        running_max = cumulative.cummax()
        dd = (cumulative - running_max) / running_max
        mdd = dd.min()

        # Sharpe Ratio
        excess = returns - (risk_free_rate / 252)
        std = excess.std()
        sharpe = (excess.mean() / std) * (252 ** 0.5) if std != 0 else 0

        return total_ret, cagr, mdd, sharpe

    # Calculate metrics
    st_tot, st_cagr, st_mdd, st_sharpe = get_metrics(results[col_strat_ret], results['Cumulative_Strategy'])
    mk_tot, mk_cagr, mk_mdd, mk_sharpe = get_metrics(results[col_mkt_ret], results['Cumulative_Market'])

    # 3. Calculate Trade Stats (Win Rate, Profit Factor)
    trades = []
    in_pos = False
    entry_p = 0

    # Iterate through rows to simulate trades
    for idx, row in results.iterrows():
        # ENTRY Logic
        if row['Signal'] == 1 and not in_pos:
            entry_p = row['close']
            in_pos = True
        # EXIT Logic
        elif (row['Signal'] == 0 and in_pos) or (idx == results.index[-1] and in_pos):
            exit_p = row['close']
            pnl = (exit_p - entry_p) / entry_p
            trades.append(pnl)
            in_pos = False

    trades_arr = np.array(trades)
    num_trades = len(trades_arr)

    # Calculate Trade Metrics
    win_rate = np.sum(trades_arr > 0) / num_trades if num_trades > 0 else 0
    gross_prof = np.sum(trades_arr[trades_arr > 0])
    gross_loss = abs(np.sum(trades_arr[trades_arr <= 0]))
    profit_factor = gross_prof / gross_loss if gross_loss > 0 else 999

    # 4. PRINT REPORT
    print("\n" + "="*60)
    print(f"{'DEMA STRATEGY PERFORMANCE':^60}")
    print("="*60)
    print(f"{'METRIC':<20} | {'DEMA STRATEGY':<15} | {'BUY & HOLD':<15}")
    print("-" * 60)
    print(f"{'Total Return':<20} | {st_tot:>14.2%} | {mk_tot:>14.2%}")
    print(f"{'CAGR':<20} | {st_cagr:>14.2%} | {mk_cagr:>14.2%}")
    print(f"{'Max Drawdown':<20} | {st_mdd:>14.2%} | {mk_mdd:>14.2%}")
    print(f"{'Sharpe Ratio':<20} | {st_sharpe:>14.2f} | {mk_sharpe:>14.2f}")
    print("-" * 60)
    print(f"{'TRADE STATISTICS':^60}")
    print("-" * 60)
    print(f"Total Trades      : {num_trades}")
    print(f"Win Rate          : {win_rate:.2%}")
    print(f"Profit Factor     : {profit_factor:.2f}")
    print("="*60 + "\n")

    # 5. PLOT CHARTS
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

    # Top Chart: Equity Curve
    ax1.plot(results.index, results['Cumulative_Strategy'], label='DEMA Strategy', color='purple', linewidth=2)
    ax1.plot(results.index, results['Cumulative_Market'], label='Buy & Hold', color='gray', linestyle='--', alpha=0.6)
    ax1.set_title('Equity Curve: DEMA vs Market', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Growth (Start=1.0)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Bottom Chart: Underwater Plot
    dd_strat = (results['Cumulative_Strategy'] - results['Cumulative_Strategy'].cummax()) / results['Cumulative_Strategy'].cummax()
    dd_mkt = (results['Cumulative_Market'] - results['Cumulative_Market'].cummax()) / results['Cumulative_Market'].cummax()

    ax2.plot(results.index, dd_strat, label='DEMA Drawdown', color='red', linewidth=1)
    ax2.fill_between(results.index, dd_strat, 0, color='red', alpha=0.2)
    ax2.plot(results.index, dd_strat, label='DEMA Drawdown', color='red', linewidth=1)
    ax2.fill_between(results.index, dd_strat, 0, color='red', alpha=0.2)
    ax2.plot(results.index, dd_mkt, label='Market Drawdown', color='gray', linestyle='--', alpha=0.3)

    ax2.set_title('Underwater Plot (Drawdown)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Drawdown %')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

evaluate_dema_vs_market(results_dema)

## WHIPSAWS
- Giảm độ trễ (Lag) $\rightarrow$ Tăng độ nhiễu (Noise/Whipsaw)
- DEMA được tạo ra để triệt tiêu độ trễ $\rightarrow$ Hệ quả tất yếu là nó cực kỳ ồn (Noisy)
- Hiện tượng DEMA cắt lên cắt xuống liên tục là "Chatter" (Sự rung lắc của tín hiệu).

### Average Directional Index (ADX)

The **Average Directional Index (ADX)** measures the **strength of a trend**, regardless of its direction.
It is part of the **Directional Movement System** developed by *Welles Wilder* and is commonly used as a **regime filter** in trend-following strategies.

---

#### True Range (TR)

The True Range captures the actual price volatility at time \( t \):

$$
TR_t = \max
\begin{cases}
High_t - Low_t, \\
|High_t - Close_{t-1}|, \\
|Low_t - Close_{t-1}|
\end{cases}
$$

---

#### Directional Movement (DM)

First, define upward and downward movements:

$$
UpMove_t = High_t - High_{t-1}
$$
$$
DownMove_t = Low_{t-1} - Low_t
$$

Then compute positive and negative directional movements:

$$
+DM_t =
\begin{cases}
UpMove_t, & \text{if } UpMove_t > DownMove_t \text{ and } UpMove_t > 0 \\
0, & \text{otherwise}
\end{cases}
$$

$$
-DM_t =
\begin{cases}
DownMove_t, & \text{if } DownMove_t > UpMove_t \text{ and } DownMove_t > 0 \\
0, & \text{otherwise}
\end{cases}
$$

---

#### Wilder’s Smoothing

Using a smoothing period \( n \) (typically \( n = 14 \)):

$$
TR^{(n)}_t = TR^{(n)}_{t-1} - \frac{TR^{(n)}_{t-1}}{n} + TR_t
$$

$$
+DM^{(n)}_t = +DM^{(n)}_{t-1} - \frac{+DM^{(n)}_{t-1}}{n} + +DM_t
$$

$$
-DM^{(n)}_t = -DM^{(n)}_{t-1} - \frac{-DM^{(n)}_{t-1}}{n} + -DM_t
$$

---

#### Directional Indicators

The directional indicators are defined as:

$$
+DI_t = 100 \times \frac{+DM^{(n)}_t}{TR^{(n)}_t}
$$

$$
-DI_t = 100 \times \frac{-DM^{(n)}_t}{TR^{(n)}_t}
$$

---

#### Directional Index (DX)

$$
DX_t = 100 \times
\frac{|+DI_t - -DI_t|}{+DI_t + -DI_t}
$$

---

#### Average Directional Index (ADX)

The ADX is the Wilder-smoothed average of the DX:

$$
ADX_t = \frac{(ADX_{t-1} \cdot (n - 1)) + DX_t}{n}
$$

---

#### Interpretation

- **ADX < 20**: weak or no trend (sideways market)
- **ADX 20–25**: trend may be forming
- **ADX > 25**: strong trend
- ADX measures **trend strength only**, not direction

Therefore, ADX is best used as a **filter** to decide whether trend-following strategies (e.g., moving average crossovers) should be applied.


In [ ]:
def calculate_adx_wilder(df, window=14):
    """
    Calculates ADX using J. Welles Wilder's standard formula.
    """
    data = df.copy()

    # 1. Calculate True Range (TR) and Directional Movement (DM)
    data['H-L'] = data['high'] - data['low']
    data['H-PC'] = abs(data['high'] - data['close'].shift(1))
    data['L-PC'] = abs(data['low'] - data['close'].shift(1))

    # TR is the max of the three calculated differences
    data['TR'] = data[['H-L', 'H-PC', 'L-PC']].max(axis=1)

    # Calculate raw movement
    data['UpMove'] = data['high'] - data['high'].shift(1)
    data['DownMove'] = data['low'].shift(1) - data['low']

    # Determine +DM and -DM
    data['+DM'] = np.where((data['UpMove'] > data['DownMove']) & (data['UpMove'] > 0), data['UpMove'], 0)
    data['-DM'] = np.where((data['DownMove'] > data['UpMove']) & (data['DownMove'] > 0), data['DownMove'], 0)

    # 2. Wilder's Smoothing Function
    # Formula: Smooth_t = (Smooth_t-1 * (n-1) + Value_t) / n
    # In Pandas, this is equivalent to .ewm(alpha=1/window, adjust=False)
    alpha = 1 / window

    data['TR_Smooth'] = data['TR'].ewm(alpha=alpha, adjust=False).mean()
    data['+DM_Smooth'] = data['+DM'].ewm(alpha=alpha, adjust=False).mean()
    data['-DM_Smooth'] = data['-DM'].ewm(alpha=alpha, adjust=False).mean()

    # 3. Calculate Directional Indicators (+DI and -DI)
    # 100 times the smoothed directional movement divided by smoothed true range
    data['+DI'] = 100 * (data['+DM_Smooth'] / data['TR_Smooth'])
    data['-DI'] = 100 * (data['-DM_Smooth'] / data['TR_Smooth'])

    # 4. Calculate DX and ADX
    # DX is the ratio of the absolute difference between +DI and -DI to the sum of +DI and -DI
    data['DX'] = 100 * abs(data['+DI'] - data['-DI']) / (data['+DI'] + data['-DI'])

    # ADX is the smoothed average of DX
    data['ADX'] = data['DX'].ewm(alpha=alpha, adjust=False).mean()

    return data

# --- PLOT ILLUSTRATION CHART ---
try:
    # Calculate ADX
    df_adx = calculate_adx_wilder(df)

    # Slice the last 300 days for better visibility
    subset = df_adx #.tail(300)

    # Create subplot with 2 rows: Price on top (larger), ADX on bottom (smaller)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

    # --- Chart 1: Price ---
    ax1.plot(subset.index, subset['close'], color='black', alpha=0.7, label='Price')
    ax1.set_title('Price Action vs Market Regimes (Trend/Sideways)')
    ax1.set_ylabel('Price')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left')

    # --- Chart 2: ADX Indicator ---
    ax2.plot(subset.index, subset['ADX'], color='purple', linewidth=2, label='ADX (14)')

    # Plot thresholds 20 and 25
    ax2.axhline(25, color='green', linestyle='--', label='Strong Trend Zone (>25)')
    ax2.axhline(20, color='red', linestyle='--', label='Sideways Zone (<20)')

    # Fill background to distinguish regimes
    # Green Area: Strong Trend Mode (Suitable for Trend Following strategies like DEMA)
    ax2.fill_between(subset.index, 25, subset['ADX'], where=(subset['ADX'] >= 25), color='green', alpha=0.2)

    # Red Area: Sideways Mode (Suitable for Mean Reversion or Cash)
    ax2.fill_between(subset.index, 0, 20, where=(subset['ADX'] <= 20), color='red', alpha=0.1)

    ax2.set_ylabel('ADX Value')
    ax2.set_ylim(0, 60) # Limit Y-axis for better readability
    ax2.legend(loc='upper left')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

except Exception as e:
    print("Error:", e)